# GPU 硬件与多卡互联

> Kernel 中的数据布局、矩阵指令和线程分工都依赖具体的 GPU 存储与计算结构。一次矩阵计算会经过多层数据通路：模型权重从 HBM 进入片上 SRAM，再由 Tensor Core 完成乘加；跨卡结果则要经过 NVLink、PCIe 或节点间网络，最慢的一段通常决定整体速度。
>
> **单卡内部**：HBM 容量大，负责保存模型与缓存；SRAM 容量小、带宽高，负责暂存当前 Tile；SM 与 Tensor Core 执行并行计算。
>
> **多卡之间**：NVLink 与 NVSwitch 连接节点内 GPU，PCIe 连接主机与设备，InfiniBand 或 RoCE 连接不同节点。通信库根据拓扑选择数据路径。
>
> 硬件数字最终会反馈到**并行策略**：频繁通信的 Tensor Parallelism 通常放在高速互联范围内，跨节点更多使用 Pipeline 或 Data Parallelism。

一次矩阵乘法开始前，权重通常先从容量较大的 HBM 搬到片上 SRAM，再由 Tensor Core 完成乘加。计算结束后，结果可能写回 HBM，也可能继续通过 NVLink 或节点间网络发送到另一张 GPU。

每一段通路的容量和带宽都不同。HBM 能保存完整模型，但距离计算单元较远；SRAM 容量很小，却适合暂存当前 Tile；跨卡互联又决定集合通信能够多快完成。

下面从单张 GPU 的两级存储开始，再扩展到 SM、Tensor Core、NVLink、PCIe 与节点间网络，观察硬件限制怎样反过来影响 Kernel 和并行策略。

## 0. GPU 存储层次

先把「GPU 怎么存数据」这件事讲清楚。GPU 芯片里并不是只有一种存储，而是两块，容量和速度差得非常远：

- **HBM**（High Bandwidth Memory，高带宽内存）：容量大、读写相对慢，负责装下整个模型；
- **SRAM**（Static RAM，静态随机存取内存）：嵌在芯片内部的一小块超高速度存储，负责在计算时快速暂存中间结果。

为什么需要两块？因为「大」和「快」在物理上是矛盾的——做不出容量又大速度又快的存储，只能权衡。于是硬件把容量堆在 HBM，把速度堆在 SRAM。

一个计算跑得快还是慢，很多时候不取决于「算得多快」，而取决于「数据在 HBM 和 SRAM 之间搬了多少趟」。把这一条记住，后面所有内容都能对上号。

### HBM 与 SRAM

下面两个词这一节会反复出现，先给定义：

- **HBM**（High Bandwidth Memory，高带宽内存）：就是常说的「显存」。容量大（一块 H100 有 80 GB），但读写速度相对慢。
- **SRAM**（Static RAM，静态随机存取内存）：嵌在 GPU 芯片里的一小块存储（H100 上只有约 20~30 MB），但读写速度极快。

一句话记忆：**HBM 负责装得下（大而慢），SRAM 负责算得快（小而快）。**

## 1. HBM 与 SRAM 的容量

先看数据。下表是近几年几款主流数据中心 GPU 的官方规格。**不要背数字**，只需要感受两个趋势：
容量在涨、速度在涨，但两块存储的「分工」没变——永远是一大一小、一慢一快。

In [ ]:
# === 主流数据中心 GPU 的内存配置 ===
# 数据来源：NVIDIA 官方规格表
# 每一行是：(型号, HBM 容量 GB, HBM 带宽 TB/s, SRAM 容量 MB, 发布年份)

gpus = [
    ("A100 40GB",  40, 1.55,  20, 2020),
    ("A100 80GB",  80, 2.00,  20, 2021),
    ("H100 SXM",   80, 3.35,  20, 2022),
    ("H200 SXM",  141, 4.80,  20, 2024),
    ("B200",      192, 8.00,  40, 2024),
]

print(f"{'型号':<14} {'HBM(GB)':>9} {'HBM带宽(TB/s)':>15} {'SRAM(MB)':>10} {'年份':>6}")
print("-" * 60)
for name, hbm_gb, hbm_bw, sram_mb, year in gpus:
    print(f"{name:<14} {hbm_gb:>9} {hbm_bw:>15.2f} {sram_mb:>10} {year:>6}")

print()
print("关键观察 1：HBM 容量从 40 GB 涨到 192 GB，涨了快 5 倍。")
print("关键观察 2：但 SRAM 一直只有 20~40 MB——片上缓存永远很小。")
print("意义：模型装得下（靠 HBM），但算得快不快（靠 SRAM 用得好不好）。")

### 1.1 HBM 与 SRAM 的带宽差异



容量之外，更关键的是**带宽**——每秒能从存储里搬多少数据。

H100 的 HBM 带宽是 3.35 TB/s（每秒 3.35 万亿字节），已经很夸张了。
但 SRAM 的带宽大约是 30 TB/s，是 HBM 的 **9 倍左右**。

这意味着什么？同样的数据，放在 SRAM 里取用比放在 HBM 里取用快近一个数量级。

一个计算如果反复去 HBM 搬数据，就会被 HBM 的速度卡住；把数据留在 SRAM 里算，就能用上约 10 倍的速度。后面 Flash Attention 的优化思路，正是围绕这一条展开的。



In [ ]:
# === HBM vs SRAM 带宽对比（以 H100 为例） ===
hbm_bw_tb = 3.35    # H100 HBM 带宽（TB/s）
sram_bw_tb = 30.0   # H100 SRAM 估算带宽（TB/s，约值）

ratio = sram_bw_tb / hbm_bw_tb
print(f"H100 HBM 带宽:  {hbm_bw_tb:.2f} TB/s")
print(f"H100 SRAM 带宽: {sram_bw_tb:.2f} TB/s（估算）")
print(f"差距：SRAM 是 HBM 的 {ratio:.1f} 倍")
print()
print("关键观察：一个计算如果反复去 HBM 搬数据，就会被 HBM 的速度卡住；")
print("如果把数据留在 SRAM 里算，就能用上 10 倍的速度。")

### 1.2 Attention 的 SRAM 复用

光说数字没感觉，我们来算一笔真实的账。

Transformer 的 attention 计算中，要生成一个「分数矩阵」，大小是 **序列长度 N × 序列长度 N**。
假设序列长度 N = 8192，有 32 个 head，用 BF16（每个数 2 字节）存。这个矩阵多大？

In [ ]:
# === 手算：attention 的 N×N 分数矩阵有多大 ===
N = 8192        # 序列长度
num_heads = 32  # head 数量
bf16 = 2        # BF16 每个数占 2 字节

# 分数矩阵形状：[head 数, N, N]，共 num_heads * N * N 个数
num_elements = num_heads * N * N
total_bytes = num_elements * bf16

print(f"分数矩阵形状: [{num_heads}, {N}, {N}]")
print(f"元素个数:     {num_elements:,}")
print(f"占用字节:     {total_bytes / 1e9:.2f} GB")
print()
# 朴素实现：这个矩阵要「写到 HBM」，softmax 时再「读回来」
# 一次写 + 一次读 = 两趟搬运
round_trip_bytes = 2 * total_bytes
hbm_bw = 3.35e12  # H100 HBM 带宽 3.35 TB/s

t = round_trip_bytes / hbm_bw
print(f"朴素实现：写一次 + 读一次，共搬运 {round_trip_bytes / 1e9:.2f} GB")
print(f"走 HBM 要花 {t * 1000:.2f} 毫秒——只是这一个矩阵，只是一个 head 的活。")
print()
print("关键观察：N 变大一点点，这个矩阵就平方级暴涨。")
print("Flash Attention 的思路：把矩阵切成小块，在 SRAM 里算完，")
print("不把整张 N×N 写回 HBM。这就是充分利用 SRAM 的思路。")

### 1.3 HBM 与 SRAM 的比较

- GPU 里有两块存储：HBM（大而慢）和 SRAM（小而快）；
- SRAM 带宽约是 HBM 的 10 倍；
- 让中间结果尽量留在 SRAM，是高性能计算的核心思路（Flash Attention 就是例子）。

接下来看看「算」的一侧。

## 2. GPU 计算单元

存储讲完了，再简单看一眼「算」的一侧。你不需要会写 CUDA，只要知道两个词：

- **SM**（Streaming Multiprocessor，流式多处理器）：GPU 是由一堆 SM 组成的，每个 SM 内部都有一块 SRAM。H100 有 132 个 SM。
- **Warp**：GPU 里最小的执行单位，固定 **32 个线程一组**。GPU 调度的时候，一次让一个 warp 的 32 个线程同时干活。

In [ ]:
# === H100 的计算资源：多少 SM，每个 SM 多大 ===
sm_count = 132                  # SM（流式多处理器）数量
shared_mem_per_sm_kb = 256      # 每个 SM 里的共享内存（shared memory）大小
register_per_sm_kb = 256        # 每个 SM 的寄存器大小

print(f"H100 的 SM 数量:            {sm_count}  （132 个 SM）")
print(f"每个 SM 的共享内存:         {shared_mem_per_sm_kb} KB")
print(f"每个 SM 的寄存器:           {register_per_sm_kb} KB")
print(f"全部共享内存加起来:         {sm_count * shared_mem_per_sm_kb / 1024:.1f} MB")
print()
print(f"Warp 大小固定 32 线程（NVIDIA 所有架构都一样）。")
print(f"关键观察：SRAM 总量 33 MB，和第一节说的 20~30 MB 对上了。")

### 2.1 Tensor Core

神经网络里最频繁的操作是矩阵乘法。GPU 里有一批专门的硬件叫 **Tensor Core**，
只干一件事：小矩阵的乘加。它比通用计算单元快得多，现代训练几乎全靠它。

Tensor Core 还决定了一件重要的事：**你用多少「位」来表示一个数**。

- 16 位（FP16 / BF16）：一个数占 2 字节；
- 8 位（FP8）：一个数占 1 字节，显存和搬运量减半，Tensor Core 算力翻倍。

位数减半听起来很划算，但代价是数字能表示的范围变小、精度变低。
所以工业界要在合适的地方做「缩放」，把大数小数都放进 8 位里。
训练大模型时用 FP8 能省一半显存、快一倍，这就是为什么它越来越流行。

FP8 有两个子格式，取舍正好相反：

| 格式 | 符号 | 指数 | 尾数 | 合适场景 |
|:---|:---:|:---:|:---:|:---|
| E4M3 | 1 | 4 | 3 | 前向计算（权重）——精度高、范围小 |
| E5M2 | 1 | 5 | 2 | 反向计算（梯度）——范围大、精度低 |

BF16 占 2 字节，FP8 占 1 字节 → 显存减半、带宽减半、算力翻倍。

关键观察：同样的模型，用 FP8 存只有一半大；但 8 位表示的范围小，需要缩放因子，这是后面量化那节的细节。


## 3. GPU 互联

单张 GPU 再强也有极限：显存放不下一个 70B 的模型，算力也不够快。于是要上多卡。

但多卡不是把卡插在一起就能快。**卡和卡之间怎么传数据，决定了多卡能快多少。**

GPU 之间的数据通路有几种，速度差异很大：

| 通路 | 一句话 | 覆盖范围 |
|:---|:---|:---|
| PCIe | 通用总线，硬盘、网卡、显卡都插在上面，慢 | 节点内 |
| NVLink | NVIDIA 专为 GPU 间通信设计的直连，快 | 节点内 |
| NVSwitch | 交换机，让所有卡接到它，任意两卡都能全速通信 | 节点内 |
| InfiniBand | 专为高性能计算设计的网络，用于跨服务器通信 | 跨节点 |

下面我们一个一个看，先看带宽排名。

In [ ]:
# === 多卡互联带宽量级对比 ===
# 数值取自 NVIDIA 公开规格（典型值）
# 注意：GB/s 是每秒十亿字节，数字越大越快

interconnects = [
    # (名称, 带宽 GB/s, 覆盖范围, 典型用途)
    ("PCIe 4.0 x16",      32,  "节点内",  "GPU-CPU、磁盘、低端 GPU-GPU"),
    ("PCIe 5.0 x16",      64,  "节点内",  "H100 与 CPU 互联"),
    ("NVLink 3.0",       300,  "节点内",  "A100 之间直连"),
    ("NVLink 4.0",       450,  "节点内",  "H100 之间直连"),
    ("NVLink 5.0",       900,  "节点内",  "B200 之间直连"),
    ("InfiniBand NDR",    50,  "跨节点",  "400 Gbps 网卡（单链路）"),
    ("InfiniBand XDR",   100,  "跨节点",  "800 Gbps 网卡（单链路）"),
]

print(f"{'名称':<20} {'带宽(GB/s)':>12} {'覆盖':>8}  典型用途")
print("-" * 80)
for name, bw, scope, use in interconnects:
    print(f"{name:<20} {bw:>12} {scope:>8}  {use}")

print()
print("关键观察 1：节点内 NVLink 是 PCIe 的 6~10 倍。")
print("关键观察 2：跨节点的 InfiniBand 比节点内 NVLink 慢 4~9 倍。")
print("含义：同一台机器里的两张卡，比跨机器的两张卡通信快得多。")

### 3.1 70B 模型的数据传输时间

光看数字不够直观。我们来算：要把一个 70B 参数的模型（BF16，约 140 GB）从一张卡搬到另一张卡，
走 NVLink 和走 PCIe 各要多久？

这个「搬模型」的动作，训练里每天都在发生：同步梯度、传输参数、迁移模型……所以它不是纸上谈兵。

In [ ]:
# === 手算：搬一个 70B 模型，NVLink vs PCIe ===
P = 70e9          # 模型参数个数（700 亿）
bf16_bytes = 2    # BF16 每个参数占 2 字节
param_bytes = P * bf16_bytes   # 总字节数 ≈ 140 GB

nvlink_bw = 450    # NVLink 4.0 双向带宽 GB/s
pcie_bw = 64       # PCIe 5.0 x16 带宽 GB/s

t_nvlink = param_bytes / (nvlink_bw * 1e9)
t_pcie = param_bytes / (pcie_bw * 1e9)

print(f"70B 模型 BF16 参数共 {param_bytes / 1e9:.0f} GB")
print(f"走 NVLink 4.0: {t_nvlink:.2f} 秒")
print(f"走 PCIe 5.0:  {t_pcie:.2f} 秒")
print(f"速度差: {t_pcie / t_nvlink:.1f} 倍")
print()
print("关键观察：同样的数据，走的通道不同，时间差 7 倍。")
print("训练中每次梯度同步都在搬这个量级的数据，所以通道选错 = 每步都慢 7 倍。")

### 3.2 NVLink 与 PCIe

PCIe 是**通用总线**：硬盘、网卡、显卡都插在上面。它设计时就要兼容各种设备，
协议开销大，传输要经过控制器、多个中转，所以慢。

NVLink 是 NVIDIA **专门给 GPU 之间通信设计的**：两颗 GPU 用信号线直接连起来，
绕开 PCIe 控制器，所以更快。但它也更「专」——只能给 GPU 之间用。

一个细节：H100 的 NVLink 4.0 双向带宽 450 GB/s，是 4 条 NVLink 通道加起来的。
我们不需要记这个，只需要记住结论：**节点内传数据，优先走 NVLink。**

### 3.3 NVSwitch

现在问题来了：一台机器里放 8 张卡，如果每两张卡之间都拉一条直连线，
8 张卡两两相连需要 28 条线（8 选 2 的组合），又乱又贵。

NVSwitch 的解决办法：**放一台交换机**。8 张卡各自把线插到交换机上，
任意两张卡之间都通过交换机转发。这样每张卡只出 1 条线，却能和其他 7 张卡全速通信。

8 张 H100 插到 4 个 NVSwitch 上，就形成了「全互联」——任意两张卡之间都有 NVLink 全速带宽。

### 3.4 InfiniBand

节点内的问题解决了，节点外呢？一台机器最多 8~16 张卡，训练千卡规模得用几十上百台机器。

机器之间用什么连？普通的以太网延迟高、不稳定，不适合大规模训练。于是有 **InfiniBand（IB）**：
为高性能计算设计的专用网络，带宽高（单链路 400 Gbps = 50 GB/s）、延迟低、稳定。

每台机器通常配 8 张 IB 网卡，每张 GPU 对应 1 张，也就是每张 GPU 都有独立的跨节点通路。
跨机器传数据时，数据从 GPU 显存出发，经过 IB 网卡直接到另一台机器的 GPU 显存，
甚至不需要经过 CPU 内存中转（这叫 GPU Direct RDMA，知道名字即可）。

来算一下跨节点搬 70B 模型要多久。

In [ ]:
# === 手算：跨节点搬 70B 模型 ===
ib_ndr_gbps = 400          # 单条 IB 网卡速率（Gbps）
ib_ndr_gbs = ib_ndr_gbps / 8   # 换算成 GB/s：1 字节 = 8 位
P = 70e9
param_bytes = P * 2        # BF16

n_hca = 8                  # 一台机器通常 8 张 IB 网卡，每张 GPU 配 1 张
agg_bw = n_hca * ib_ndr_gbs   # 8 张网卡一起用，聚合带宽

t_single = param_bytes / (ib_ndr_gbs * 1e9)
t_agg = param_bytes / (agg_bw * 1e9)

print(f"IB 单链路: {ib_ndr_gbs:.0f} GB/s；8 链路聚合: {agg_bw:.0f} GB/s")
print(f"跨节点搬 70B 模型 ({param_bytes / 1e9:.0f} GB)：")
print(f"  单条链路: {t_single:.2f} 秒")
print(f"  8 条一起: {t_agg:.2f} 秒")
print()
print("对比：节点内 NVLink 搬同样数据只要 0.31 秒。")
print("关键观察：即使 8 条 IB 全用上，跨节点仍比节点内慢 1 倍以上。")

## 4. DGX H100 的硬件结构

上面每一样东西都单独讲了，现在把它们装进一台真实机器。

**DGX H100** 是 NVIDIA 官方的高密度训练机，一个机箱里塞 8 张 H100，是这个时代的「标准训练单元」。
它长这样：

- 8 张 H100 SXM，每张 80 GB 显存；
- 4 个 NVSwitch，让 8 张卡两两之间全互联（任意两张卡 450 GB/s 双向）；
- 8 张 IB 网卡，每张 GPU 对应 1 条跨节点通路；
- 2 颗 CPU 负责管理和数据加载（走 PCIe）。

一句话：**节点内靠 NVSwitch 全互联，节点外靠 IB 网卡。** 这就是现代训练集群的基本结构。

In [ ]:
# === 节点内 vs 跨节点：带宽差多少 ===
# 节点内：任意两张卡之间 NVLink 450 GB/s，8 卡共 C(8,2)=28 对
intra_link = 450
n_pairs = 8 * 7 // 2        # 28 对
intra_agg = intra_link * n_pairs

# 跨节点：8 张 IB 网卡，每张 50 GB/s
ib_link = 50
n_hca = 8
inter_agg = ib_link * n_hca

print(f"节点内全互联总带宽: {intra_agg:,} GB/s（28 对 × 450）")
print(f"跨节点聚合带宽:     {inter_agg} GB/s（8 条 × 50）")
print()
print(f"差 {intra_agg / inter_agg:.1f} 倍！")
print()
print("关键观察：这个 30 倍的差距，是所有并行策略选择的底层原因。")
print("通信多的操作要尽量留在节点内，跨节点通信要尽量少。")

## 5. 硬件约束与训练方案

现在我们知道：节点内任意两卡 450 GB/s，跨节点只有 50 GB/s，差 9 倍（单链路）以上。

分布式训练里，不同并行策略的「通信习惯」完全不同：

- **Tensor Parallelism（切权重）**：每一层的前向、反向都要跨卡同步数据，通信极其频繁。
  → **必须放同一个节点内**，靠 NVLink。
- **Pipeline Parallelism（切层）**：只有相邻两层的卡之间传一次激活值，通信很少。
  → 可以跨节点。
- **数据并行 DDP / FSDP**：每步只在反向结束时同步一次梯度。
  → 节点内、跨节点都行，跨节点也能靠 IB 的聚合带宽。
- **MoE 专家并行**：每个 token 要发到「持有它对应专家」的卡上，通信量大且模式特殊。
  → 对拓扑最敏感，能放节点内就放节点内。

这些名字现在看不懂没关系，下一节（集合通信）和 5D 并行那节会逐个展开。
这里只需要记住一句话：**通信越频繁、越大的操作，越要放在带宽高的地方（节点内）。**

In [ ]:
# === 不同并行策略对带宽的敏感程度 ===
strategies = [
    # (策略, 通信模式, 通信频繁度, 拓扑要求)
    ("Tensor Parallelism",   "每层都同步",      "非常频繁", "必须节点内 (NVLink)"),
    ("Pipeline Parallelism", "相邻层传激活值",  "很少",      "可跨节点"),
    ("DDP / FSDP",           "每步同步一次梯度", "少",      "节点内/跨节点都行"),
    ("MoE 专家并行",          "每步 all-to-all", "频繁",     "优先节点内"),
]

print(f"{'策略':<24}{'通信模式':<20}{'频繁度':<12}拓扑要求")
print("-" * 78)
for s, mode, freq, req in strategies:
    print(f"{s:<24}{mode:<20}{freq:<12}{req}")
print()
print("关键观察：通信越频繁，越依赖高速连接（NVLink）。")

## 6. GPU 拓扑检查

理论说完了，来看实际操作。多卡训练出问题（比如莫名慢），第一个要查的就是拓扑。

最常用的命令是：

```bash
nvidia-smi topo -m
```

它会输出一张 8×8 的矩阵：第 i 行第 j 列表示「第 i 张卡到第 j 张卡怎么连」。
下面是一个 8 卡 H100 节点的典型输出（在 Jupyter 里用字符串模拟，真实机器上直接运行命令即可）：

In [ ]:
# === nvidia-smi topo -m 输出示例（8 卡 H100 节点） ===
# 真实机器上运行: nvidia-smi topo -m
# 这里用字符串模拟典型输出

topo_output = """
        GPU0    GPU1    GPU2    GPU3    GPU4    GPU5    GPU6    GPU7
GPU0     X      NV12    NV12    NV12    NV12    NV12    NV12    NV12
GPU1    NV12     X      NV12    NV12    NV12    NV12    NV12    NV12
GPU2    NV12    NV12     X      NV12    NV12    NV12    NV12    NV12
GPU3    NV12    NV12    NV12     X      NV12    NV12    NV12    NV12
GPU4    NV12    NV12    NV12    NV12     X      NV12    NV12    NV12
GPU5    NV12    NV12    NV12    NV12    NV12     X      NV12    NV12
GPU6    NV12    NV12    NV12    NV12    NV12    NV12     X      NV12
GPU7    NV12    NV12    NV12    NV12    NV12    NV12    NV12     X

Legend:
  X    = Self
  SYS  = 跨 NUMA 节点（最慢）
  PIX  = 同一个 PCIe switch（走 PCIe，较慢）
  PHB  = 跨 PCIe Host Bridge（较慢）
  NV#  = N 条 NVLink 通道（快）
"""

print(topo_output)
print("怎么读：")
print("  满屏 NV12 = 8 张卡之间都有 12 条 NVLink，健康，跑得快。")
print("  如果出现大片 PIX / SYS = NVLink 没工作，通信只能走 PCIe，会慢很多。")
print()
print("关键观察：这张图就是「拓扑感知调度」的依据——")
print("调度器把通信多的任务排在 NV12 的卡上。")

### 6.1 常用硬件检查命令

- `nvidia-smi`：看每张卡的显存占用、温度、利用率（相当于显卡版的任务管理器）。
- `watch -n 1 nvidia-smi`：每秒刷新一次，观察训练时显存变化。
- `nvtop`：更友好的交互式监控界面（需要单独安装）。
- nccl-tests（`all_reduce_perf`）：实测卡间通信带宽。如果实测远低于规格，
  多半是 NVLink 没启用或驱动/固件问题。

## 小结

这一节信息量不小，但记住一张「存储与互联」表即可：

| 概念 | 要点 |
|:---|:---|
| HBM | 大而慢，容量靠它 |
| SRAM | 小而快，速度靠它 |
| SM / Warp | GPU 的组织方式 |
| Tensor Core | 位数越少越快 |
| PCIe | 通用但慢 |
| NVLink | 节点内快通道 |
| NVSwitch | 让任意两卡全速通信 |
| InfiniBand | 跨机器通信 |

这一节所学的内容：确认你已经搞懂：

- [ ] HBM 是显存（大而慢），SRAM 是片上小存储（小而快），带宽差约 10 倍
- [ ] 中间结果留在 SRAM 是关键，Flash Attention 就是典型例子
- [ ] Tensor Core 用更少的位数（FP8 vs BF16）能省显存、翻倍算力
- [ ] NVLink（节点内）比 PCIe 快 6~10 倍，比跨节点 IB 慢 4~9 倍
- [ ] NVSwitch 让节点内任意两张卡都全速互联
- [ ] DGX H100 节点内 8 卡全互联，跨节点靠 IB 网卡
- [ ] 通信越频繁的并行策略，越必须放节点内
- [ ] `nvidia-smi topo -m` 是排查多卡性能问题的第一命令

## 作业

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

**作业 1：算一算 attention 分数矩阵**

序列长度 N = 8192，head 数 h = 32，BF16（2 字节）。朴素 attention 要先把
分数矩阵 [h, N, N] 写到 HBM，softmax 时再读回来（共两趟）。算一算：
这个矩阵占多大（GB）？两趟搬运共多少字节？

小提示：元素个数 = h × N × N，再乘 2 字节；两趟就再乘 2。

In [ ]:
# 作业 1：attention 分数矩阵的大小与搬运量
N = 8192
h = 32
bf16_bytes = 2

# TODO: 矩阵大小（GB）
mat_gb = None

# TODO: 两趟搬运总量（GB，写一次 + 读一次）
round_trip_gb = None

assert mat_gb is not None, "请先算矩阵大小"
assert round_trip_gb is not None, "请先算两趟搬运量"

expected_mat = h * N * N * bf16_bytes / 1e9
assert abs(mat_gb - expected_mat) < 0.1, f"矩阵应为 {expected_mat:.2f} GB"
assert abs(round_trip_gb - 2 * expected_mat) < 0.2, f"搬运量应为 {2 * expected_mat:.2f} GB"

print(f"作业 1 通过！")
print(f"  分数矩阵: {mat_gb:.2f} GB")
print(f"  写 + 读两趟: {round_trip_gb:.2f} GB")
print(f"  这还只是一个层、一次前向的量——序列越长，越需要留在 SRAM。")

**作业 2：跨节点 vs 节点内搬模型**

70B 模型 BF16 参数（140 GB），在两张卡之间传输。
同节点走 NVLink 4.0（450 GB/s）要多久？跨节点走 IB NDR 单链路（50 GB/s）要多久？差几倍？

小提示：时间 = 字节数 ÷ 带宽。字节数 = 70e9 × 2。

In [ ]:
# 作业 2：同节点 vs 跨节点传输时间
P = 70e9
param_bytes = P * 2       # BF16

nvlink_bw = 450           # NVLink 4.0 双向 GB/s
ib_bw = 50                # IB NDR 单链路 GB/s

# TODO: 两种传输时间（秒）
t_intra = None
t_inter = None

assert t_intra is not None and t_inter is not None, "请先计算两个时间"
expected_intra = param_bytes / (nvlink_bw * 1e9)
expected_inter = param_bytes / (ib_bw * 1e9)
assert abs(t_intra - expected_intra) < 0.01, f"节点内应为 {expected_intra:.2f} 秒"
assert abs(t_inter - expected_inter) < 0.01, f"跨节点应为 {expected_inter:.2f} 秒"

print(f"作业 2 通过！")
print(f"  节点内 NVLink: {t_intra:.2f} 秒")
print(f"  跨节点 IB:    {t_inter:.2f} 秒")
print(f"  差 {t_inter / t_intra:.1f} 倍 —— 这就是为什么通信多的任务必须留在节点内。")

**作业 3：读一张拓扑图**

在一台 8 卡机器上跑 `nvidia-smi topo -m`，输出里满屏都是 `PIX`（走 PCIe）而不是 `NV12`。
训练 DDP 模型时会观察到什么现象？应该优先检查什么？

小提示：PIX 表示两张卡通过 PCIe 相连，NVLink 没工作。NVLink 不工作通常和驱动、
CUDA 版本、主板设置有关。想想：梯度同步走 PCIe 会比走 NVLink 慢多少？

In [ ]:
# 作业 3：识别拓扑问题（概念题，填空）

# 现象：从下面三个里选一个
#   "训练速度正常，与 NVLink 一致"
#   "训练速度大幅下降，all-reduce 同步成为瓶颈"
#   "训练直接报错无法启动"
phenomenon = None

# 排查方向：从下面三个里选一个
#   "驱动/CUDA 版本是否支持 NVLink、主板 NVLink 配置是否启用"
#   "调大 batch size"
#   "换一台 GPU"
check_action = None

assert phenomenon == "训练速度大幅下降，all-reduce 同步成为瓶颈", \
    "PIX 意味着走 PCIe，比 NVLink 慢 6~10 倍，梯度同步会变成瓶颈"
assert check_action == "驱动/CUDA 版本是否支持 NVLink、主板 NVLink 配置是否启用", \
    "NVLink 不工作先从驱动、CUDA、主板配置查起"

print(f"作业 3 通过！")
print(f"  现象: {phenomenon}")
print(f"  排查: {check_action}")

## 参考资料

- NVIDIA, [H100 Tensor Core GPU Architecture Whitepaper](https://resources.nvidia.com/en-us/tensor-core), 2022
- NVIDIA, [Hopper FP8 Tensor Cores](https://www.nvidia.com/en-us/data-center/hopper-architecture/), 2022
- NVIDIA, [NVLink and NVSwitch](https://www.nvidia.com/en-us/data-center/nvlink/), 2023
- NVIDIA, [DGX H100 System Architecture](https://www.nvidia.com/en-us/data-center/dgx-h100/), 2023
- Dao et al., [FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness](https://arxiv.org/abs/2205.14135), 2022
- NVIDIA, [NCCL Tests Documentation](https://github.com/NVIDIA/nccl-tests), 2023